### Imports

In [1]:
# !pip install pandas

In [2]:
import pandas as pd
import glob

In [3]:
files = glob.glob(r"..\Downloads\DATA\vehicle_speeds_*.csv")

df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

In [4]:
df.head()

,vehicle_id,carriageway,vehicle_type,speed_kmh,speed_source,entry_timestamp_s,entry_frame,total_frames,segment_id
0,L0,left,car,13.7,bev_avg,0.24,6,3105,0
1,L1,left,car,9.4,bev_avg,0.24,6,3105,0
2,L2,left,car,38.8,bev_avg,0.24,6,3105,0
3,L3,left,car,62.2,bev_avg,0.24,6,3105,0
4,L4,left,truck,NaN,none,0.28,7,3105,0


In [5]:
# group segment ids to get cumulative frames
segment_frames = df.groupby('segment_id')['total_frames'].first().sort_index()
frames_before = segment_frames.cumsum().shift(fill_value=0)
df['frames_before_segment'] = df['segment_id'].map(frames_before)

# absolute frame for each entity
df['absolute_frame'] = df['frames_before_segment'] + df['entry_frame']


df['timestamp_seconds'] = df['absolute_frame'] / 25
df['timestamp'] = pd.to_timedelta(df['timestamp_seconds'], unit='s')

In [6]:
df.head()

,vehicle_id,carriageway,vehicle_type,speed_kmh,speed_source,entry_timestamp_s,entry_frame,total_frames,segment_id,frames_before_segment,absolute_frame,timestamp_seconds,timestamp
0,L0,left,car,13.7,bev_avg,0.24,6,3105,0,0,6,0.24,0 days 00:00:00.240000
1,L1,left,car,9.4,bev_avg,0.24,6,3105,0,0,6,0.24,0 days 00:00:00.240000
2,L2,left,car,38.8,bev_avg,0.24,6,3105,0,0,6,0.24,0 days 00:00:00.240000
3,L3,left,car,62.2,bev_avg,0.24,6,3105,0,0,6,0.24,0 days 00:00:00.240000
4,L4,left,truck,NaN,none,0.28,7,3105,0,0,7,0.28,0 days 00:00:00.280000


In [7]:
## 5 mins bins
bin_size = 60*5
max_time = df['timestamp_seconds'].max()
bins = range(0, int(max_time) + bin_size, bin_size)
labels = [f"{i*5}-{(i+1)*5}min" for i in range(len(bins)-1)]

df['time_bin'] = pd.cut(df['timestamp_seconds'], bins=bins, labels=labels, right=False)

In [8]:
df.head()

,vehicle_id,carriageway,vehicle_type,speed_kmh,speed_source,entry_timestamp_s,entry_frame,total_frames,segment_id,frames_before_segment,absolute_frame,timestamp_seconds,timestamp,time_bin
0,L0,left,car,13.7,bev_avg,0.24,6,3105,0,0,6,0.24,0 days 00:00:00.240000,0-5min
1,L1,left,car,9.4,bev_avg,0.24,6,3105,0,0,6,0.24,0 days 00:00:00.240000,0-5min
2,L2,left,car,38.8,bev_avg,0.24,6,3105,0,0,6,0.24,0 days 00:00:00.240000,0-5min
3,L3,left,car,62.2,bev_avg,0.24,6,3105,0,0,6,0.24,0 days 00:00:00.240000,0-5min
4,L4,left,truck,NaN,none,0.28,7,3105,0,0,7,0.28,0 days 00:00:00.280000,0-5min


In [9]:
# clean this shit
avg_speed_per_bin = df.groupby(['time_bin','carriageway','vehicle_type'])['speed_kmh'].mean().reset_index()
avg_speed_per_bin.rename(columns={'speed_kmh': 'avg_speed_kmh'}, inplace=True)

In [10]:
avg_speed_per_bin.head()

,time_bin,carriageway,vehicle_type,avg_speed_kmh
0,0-5min,left,car,44.024066
1,0-5min,left,truck,42.387054
2,0-5min,right,car,101.813830
3,0-5min,right,truck,106.011877
4,5-10min,left,car,52.515986
